In [2]:
import pandas as pd

#X is features
X = pd.read_parquet("../data/processed/features.parquet")

#Y is target
#0 is completed
#1 is terminated
y = pd.read_parquet("../data/processed/target.parquet")["terminated"]


df = pd.read_parquet("../data/processed/trials.parquet")
print(X.shape, y.shape)

(15461, 48) (15461,)


In [3]:
train_mask = df["start_date"].dt.year < 2018
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"train: {len(X_train)}, test: {len(X_test)}")
print(f"train termination rate: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

train: 9075, test: 6386
train termination rate: 0.111, test: 0.112


In [4]:
print(y.shape)

(15461,)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, average_precision_score

In [7]:
#model 1: logistic regression
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced") #11% positive class requires weight blanace
).fit(X_train, y_train)

In [9]:
#model 2: histogram-based gradient boosting
weights = compute_sample_weight("balanced", y_train)
hgb = HistGradientBoostingClassifier(
    max_iter=400, learning_rate=0.06,
    early_stopping=True, validation_fraction=0.15, random_state=42
).fit(X_train, y_train, sample_weight=weights)

In [11]:
for name, model in [("logreg", log_reg), ("hgb", hgb)]:
    probs = model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    print(f"{name}: ROC-AUC {roc:.3f} | PR-AUC {pr:.3f}")

logreg: ROC-AUC 0.710 | PR-AUC 0.243
hgb: ROC-AUC 0.707 | PR-AUC 0.245


In [12]:
import numpy as np
from sklearn.inspection import permutation_importance

In [13]:
importance = permutation_importance(hgb, X_test, y_test, n_repeats=10,
                             random_state=42, scoring="average_precision")
order = np.argsort(importance.importances_mean)[::-1][:15]
for i in order:
    print(f"{X.columns[i]:<28} {importance.importances_mean[i]:.4f}")

criteria_len                 0.0534
healthy_volunteers           0.0433
condition_oncology           0.0239
min_age                      0.0133
intervention_BEHAVIORAL      0.0088
max_age                      0.0081
is_masked                    0.0063
condition_metabolic          0.0057
sponsor_INDUSTRY             0.0056
countries_count              0.0022
intervention_OTHER           0.0021
sponsor_OTHER_GOV            0.0019
phase_PHASE2                 0.0018
intervention_DIETARY_SUPPLEMENT 0.0013
sex_MALE                     0.0009


In [15]:
from sklearn.metrics import classification_report, confusion_matrix

In [16]:
probs = hgb.predict_proba(X_test)[:, 1]
threshold = 0.35
preds = (probs > threshold).astype(int)

print(classification_report(y_test, preds, digits=3))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

           0      0.952     0.468     0.627      5672
           1      0.161     0.811     0.269       714

    accuracy                          0.506      6386
   macro avg      0.556     0.639     0.448      6386
weighted avg      0.863     0.506     0.587      6386

[[2653 3019]
 [ 135  579]]


In [17]:
from sklearn.metrics import confusion_matrix

for t in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    preds = (probs > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    recall = tp / (tp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    print(f"t={t}: recall {recall:.2f} | precision {precision:.2f} | "
          f"flagged {tp+fp} | caught {tp}/{tp+fn}")

t=0.35: recall 0.81 | precision 0.16 | flagged 3598 | caught 579/714
t=0.4: recall 0.73 | precision 0.18 | flagged 2930 | caught 522/714
t=0.45: recall 0.65 | precision 0.19 | flagged 2416 | caught 464/714
t=0.5: recall 0.58 | precision 0.21 | flagged 1991 | caught 413/714
t=0.55: recall 0.48 | precision 0.23 | flagged 1464 | caught 340/714
t=0.6: recall 0.33 | precision 0.27 | flagged 888 | caught 236/714


In [18]:
import joblib
joblib.dump(hgb, "../app/model.joblib")
joblib.dump(list(X.columns), "../app/feature_names.joblib")
print("saved")

saved


In [19]:
import os
print(os.listdir("../app"))

['model.joblib', 'feature_names.joblib']
